In [ ]:
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)

In [ ]:
movement_similarity_vehicles_latest=pd.read_csv("movement_similarity_vehicles_latest.csv")

In [ ]:
movement_similarity_vehicles_latest.head(2)

final_data=[]
for i,row in movement_similarity_vehicles_latest.fillna(-1).iterrows():
    new_row=[0]*226
    new_row[0]=row['vehicle_number']
    new_row[int(row['cluster1'])]=row['priority1']
    new_row[int(row['cluster2'])]=row['priority2']
    new_row[int(row['cluster3'])]=row['priority3']
    new_row[int(row['cluster4'])]=row['priority4']
    new_row[int(row['cluster5'])]=row['priority5']
    new_row[int(row['cluster6'])]=row['priority6']
    new_row[int(row['cluster7'])]=row['priority7']
    new_row[int(row['cluster8'])]=row['priority8']
    new_row[int(row['cluster9'])]=row['priority9']
    new_row[int(row['cluster10'])]=row['priority10']


    
    final_data.append(new_row)
    
    

In [ ]:
final_data=[]
for i,row in movement_similarity_vehicles_latest.fillna(-1).iterrows():
    new_row=[0]*226
    new_row[0]=row['vehicle_number']
    new_row[int(row['cluster1'])]=1
    new_row[int(row['cluster2'])]=1
    new_row[int(row['cluster3'])]=1
    new_row[int(row['cluster4'])]=1
    new_row[int(row['cluster5'])]=1
    new_row[int(row['cluster6'])]=1
    new_row[int(row['cluster7'])]=1
    new_row[int(row['cluster8'])]=1
    new_row[int(row['cluster9'])]=1
    new_row[int(row['cluster10'])]=1


    
    final_data.append(new_row)
    

In [ ]:
columns=['vehicle_number']+['cluster_{}'.format(i) for i in range(1,226)]
final_dataframe=pd.DataFrame(final_data,columns=columns)

In [ ]:
final_dataframe.drop_duplicates('vehicle_number',inplace=True)

In [ ]:
for i in final_dataframe.drop('vehicle_number',axis=1).columns:
    if final_dataframe[i].sum()==0:
        final_dataframe.drop(i,axis=1,inplace=True)

In [ ]:
final_dataframe.drop('cluster_225',axis=1,inplace=True)

# data array sanitized

In [ ]:
x=final_dataframe.drop('vehicle_number',axis=1).values

# applying Kmeans

In [ ]:
from sklearn.cluster import KMeans
import numpy as np 

In [ ]:
kmeans = KMeans(n_clusters=1000, random_state=0).fit(x)

In [ ]:
final_dataframe['Kmeans_clustered']=kmeans.labels_

# Kmeans validity

In [ ]:
movement_similarity_vehicles_latest[movement_similarity_vehicles_latest['vehicle_number'].isin(final_dataframe[final_dataframe['Kmeans_clustered']==50].vehicle_number)].cluster1.value_counts()                    




# CO-SINE similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


# Density based 

In [ ]:
from sklearn.cluster import DBSCAN

In [ ]:
clustering = DBSCAN(eps=2, min_samples=2,n_jobs=-1).fit(x)

In [ ]:
final_dataframe['DB_clustered']=clustering.labels_

In [ ]:
final_dataframe['DB_clustered'].value_counts()

In [ ]:
final_dataframe.to_csv("DB_clustered_data_eps2_minsamp_2.csv")

# density based validity

In [ ]:
movement_similarity_vehicles_latest[movement_similarity_vehicles_latest['vehicle_number'].isin(final_dataframe[final_dataframe['DB_clustered']==0].vehicle_number)].cluster2.value_counts()                


# spectral clustering

In [ ]:
from sklearn.cluster import SpectralClustering


In [ ]:
spec_clustering = SpectralClustering(n_clusters=2000,
assign_labels="discretize",
random_state=0,n_jobs=-1).fit(x)